# Phase 3 (LSTM) — full 50-fold grid

This notebook replicates the headline LSTM grid in `03_lstm_baseline.ipynb` but
uses a **true LSTM** encoder (`nn.LSTM`) instead of GRU. The architecture is
otherwise identical: bidirectional, hidden=64, single layer, AMP enabled.

Output goes to `phase3_outputs/lstm_v2/` so the existing GRU results in
`phase3_outputs/lstm/` are preserved for the architecture comparison section.

**Configuration (locked):** hidden=64, epochs=12, batch_size=64,
soft Gaussian labels with σ=5 days. Matches the GRU run that produced the
existing `lstm_results.csv`.

**Resumable:** each fold writes to CSV immediately. If the kernel dies, just
rerun and finished folds will be skipped.

**Runtime estimate:** ~70 to 100 minutes total on the MIG A100 slice for all
50 folds. The rank1 folds are 9x faster than all10 because rank1 has 9x fewer
training records.

Two side outputs:
1. `lstm_v2_results.csv` — per (stream, policy, year, phase) metrics
2. `lstm_v2_predictions.npz` — argmax predictions for downstream analysis
3. `lstm_logits.npz` — full per-day logits for the All/all10/2020 fold (for F8)


## 1. Setup

In [1]:
from pathlib import Path
import sys, os, time, gc, psutil
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'Total GPU memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def memstat(tag=''):
    cpu = psutil.Process(os.getpid()).memory_info().rss / 1e9
    if torch.cuda.is_available():
        gpu_alloc = torch.cuda.memory_allocated() / 1e9
        gpu_reserved = torch.cuda.memory_reserved() / 1e9
        return f'{tag} RAM={cpu:.2f}GB  GPU_alloc={gpu_alloc:.2f}GB  GPU_reserved={gpu_reserved:.2f}GB'
    return f'{tag} RAM={cpu:.2f}GB'

print(memstat('init:'))


torch: 2.7.1+cu118
CUDA available: True
Device: NVIDIA A100-PCIE-40GB MIG 1g.5gb
Total GPU memory: 5.10 GB
init: RAM=0.44GB  GPU_alloc=0.00GB  GPU_reserved=0.00GB


In [2]:
TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
OUTPUT_DIR  = PROJECT_ROOT / 'phase3_outputs' / 'lstm_v2'    # SEPARATE from GRU output
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === LOCKED hyperparameters (match what produced the GRU lstm_results.csv) ===
YEARS       = [2017, 2018, 2019, 2020, 2021]
STREAMS     = ['S1_only', 'S2_only', 'Met_only', 'S1+S2', 'All']
POLICIES    = ['all10', 'rank1']
EPOCHS      = 12
HIDDEN      = 64
BATCH_SIZE  = 64
LR          = 1e-3
WD          = 1e-5
USE_AMP     = True

print(f'Output dir: {OUTPUT_DIR}')
print(f'Config: epochs={EPOCHS}, hidden={HIDDEN}, batch_size={BATCH_SIZE}, amp={USE_AMP}')
print(f'Encoder: LSTM (true LSTM, not GRU)')


Output dir: /fs1/home/atikul/Phenology/phase3_outputs/lstm_v2
Config: epochs=12, hidden=64, batch_size=64, amp=True
Encoder: LSTM (true LSTM, not GRU)


## 2. Imports + memory-safe loader

In [3]:
from phase3_baselines.dataset import (load_year_bundle, SelectionPolicy,
                                       apply_selection, concat_bundles)
from phase3_baselines.baseline_gru_lstm_compare import (
    MultiStreamModel, StreamEncoderLSTM,
)
from phase3_baselines.baseline_lstm import soft_label_loss
from phase3_baselines.evaluate import evaluate_predictions
from phase3_baselines.resumable import append_row, already_done, save_pred_npz


STREAM_CONFIGS = {
    'S1_only':  ('S1',),
    'S2_only':  ('S2',),
    'Met_only': ('Met',),
    'S1+S2':    ('S1', 'S2'),
    'All':      ('S1', 'S2', 'Met'),
}


def load_loyo_for_year(test_year, policy):
    """Load 4 train years + 1 test year. Releases memory aggressively."""
    test_b = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{test_year}.npz'), policy)
    train_parts = {}
    for y in YEARS:
        if y == test_year: continue
        train_parts[y] = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
    train_b = concat_bundles(train_parts, years=[y for y in YEARS if y != test_year])
    del train_parts
    gc.collect()
    return train_b, test_b


class TensorBundle:
    """Holds tensors in CPU memory; per-batch GPU streaming to avoid OOM."""
    def __init__(self, bundle):
        self.X    = torch.from_numpy(bundle['X'])
        self.M    = torch.from_numpy(bundle['mask'])
        self.Y    = torch.from_numpy(bundle['Y'])
        self.LM   = torch.from_numpy(bundle['label_mask'])
        self.bag  = torch.from_numpy(bundle['bag_id'])
        self.rank = torch.from_numpy(bundle['rank'])
        self.yr   = torch.from_numpy(bundle['harvest_year'])
        self.xid  = torch.from_numpy(bundle['xid'])
        self.N    = self.X.shape[0]


def iter_batches(tb, batch_size, shuffle, device, generator=None):
    idx = torch.randperm(tb.N, generator=generator) if shuffle else torch.arange(tb.N)
    for i in range(0, tb.N, batch_size):
        sl = idx[i:i+batch_size]
        x = tb.X[sl].to(device, non_blocking=True)
        m = tb.M[sl].to(device, non_blocking=True).to(torch.float32)
        y = tb.Y[sl].to(device, non_blocking=True)
        lm = tb.LM[sl].to(device, non_blocking=True)
        yield x, m, y, lm


## 3. Training function (LSTM variant)

Uses `StreamEncoderLSTM` from `baseline_gru_lstm_compare`. Identical to the
GRU version in 03_lstm_baseline but with the LSTM encoder substituted.

Optionally returns the trained model + test loader so we can collect per-day
logits for F8 from the All/all10/2020 fold without retraining.


In [4]:
def train_one_fold_lstm(train_b, test_b, active, hidden=64, epochs=12,
                          batch_size=64, lr=1e-3, weight_decay=1e-5,
                          use_amp=True, device=DEVICE, verbose=False,
                          return_model=False):
    """Train one fold with LSTM encoder. Returns predictions + (optionally) model."""
    tb_tr = TensorBundle(train_b)
    tb_te = TensorBundle(test_b)
    n_phases = train_b['Y'].shape[2]
    feature_names = train_b['feature_names']

    model = MultiStreamModel(
        feature_names=feature_names,
        encoder_cls=StreamEncoderLSTM,
        hidden=hidden, num_layers=1, n_phases=n_phases, dropout=0.1,
    ).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and device.type == 'cuda'))
    rng = torch.Generator().manual_seed(0)

    for ep in range(epochs):
        model.train()
        total, nb = 0.0, 0
        for x, m, y, lm in iter_batches(tb_tr, batch_size, True, device, rng):
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(use_amp and device.type == 'cuda')):
                logits = model(x, m, active=active)
                loss = soft_label_loss(logits, y, lm)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            total += loss.item(); nb += 1
        if verbose:
            print(f'    epoch {ep+1:2d}/{epochs}: loss={total/max(nb,1):.4f}')

    # Argmax predictions
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for x, m, y, lm in iter_batches(tb_te, batch_size, False, device):
            with torch.amp.autocast('cuda', enabled=(use_amp and device.type == 'cuda')):
                logits = model(x, m, active=active)
            pred_day = logits.argmax(dim=1).cpu().numpy()
            truth_day = y.argmax(dim=1).cpu().numpy()
            truth_day[~lm.cpu().numpy()] = -1
            preds.append(pred_day); truths.append(truth_day)
    pred_all  = np.concatenate(preds)
    truth_all = np.concatenate(truths)
    xid  = tb_te.xid.numpy()
    bag  = tb_te.bag.numpy()
    rank_ = tb_te.rank.numpy()
    yr   = tb_te.yr.numpy()

    result = {
        'pred': pred_all, 'truth': truth_all,
        'xid': xid, 'bag': bag, 'rank': rank_, 'year': yr,
    }
    if return_model:
        result['model'] = model
        result['tb_te'] = tb_te
    else:
        del model, opt, scaler, tb_tr, tb_te
        gc.collect()
        if device.type == 'cuda': torch.cuda.empty_cache()
    return result


## 4. Smoke test (one fold)

In [5]:
# Quick check: All / rank1 / 2020 should be fast (~30s) and give reasonable MAE
print(memstat('before smoke:'))
policy = SelectionPolicy(use_only_rank1=True)
train_b, test_b = load_loyo_for_year(2020, policy)
print(f'train N={len(train_b["xid"])}, test N={len(test_b["xid"])}')

t0 = time.time()
result = train_one_fold_lstm(
    train_b, test_b, active=STREAM_CONFIGS['All'],
    hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    use_amp=USE_AMP, verbose=True,
)
print(f'Smoke fold completed in {time.time()-t0:.1f}s')
print(memstat('after smoke:'))

phase_ids = train_b['phase_ids']
del train_b, test_b
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

from phase3_baselines.evaluate import evaluate_predictions, print_evaluation
report = evaluate_predictions(
    pred_days=result['pred'].astype(np.float32),
    true_days=result['truth'].astype(np.float32),
    phase_ids=phase_ids, years=result['year'], ranks=result['rank'],
)
print_evaluation(report, 'LSTM smoke (All/rank1/2020)')


before smoke: RAM=0.44GB  GPU_alloc=0.00GB  GPU_reserved=0.00GB
train N=2421, test N=585
    epoch  1/12: loss=0.2761
    epoch  2/12: loss=0.1167
    epoch  3/12: loss=0.1059
    epoch  4/12: loss=0.0970
    epoch  5/12: loss=0.0858
    epoch  6/12: loss=0.0785
    epoch  7/12: loss=0.0724
    epoch  8/12: loss=0.0686
    epoch  9/12: loss=0.0672
    epoch 10/12: loss=0.0669
    epoch 11/12: loss=0.0659
    epoch 12/12: loss=0.0649
Smoke fold completed in 14.2s
after smoke: RAM=1.15GB  GPU_alloc=0.02GB  GPU_reserved=0.26GB

LSTM smoke (All/rank1/2020)

[Headline — macro over phases]
  macro_mae               : 8.730
  macro_r2                : -0.401
  macro_within_5d         : 0.408
  n_phases_evaluated      : 7

[Per phase]
         n     mae    rmse     r2  within_5d   bias
phase                                              
10     528  10.441  12.753 -0.102      0.320 -4.252
12     518  10.766  13.627 -0.037      0.315 -2.311
15     489  10.049  12.847 -0.436      0.342 -2.376
18 

## 5. Full 50-fold grid (resumable)

Order:
- Outer: streams (S1_only, S2_only, Met_only, S1+S2, All)
- Middle: policies (all10, rank1)
- Inner: test years (2017..2021)

The all10 policy has 9x more training records than rank1, so all10 folds are
~9x slower. We run rank1 first within each (stream, policy) pair to ensure the
fast folds finish in case of session timeout.

The All/all10/2020 fold gets `return_model=True` so we can save logits for F8.


In [6]:
RESULTS_CSV = OUTPUT_DIR / 'lstm_v2_results.csv'
PREDS_NPZ   = OUTPUT_DIR / 'lstm_v2_predictions.npz'
LOGITS_NPZ  = OUTPUT_DIR / 'lstm_logits.npz'

policy_map = {
    'all10': SelectionPolicy(use_only_rank1=False),
    'rank1': SelectionPolicy(use_only_rank1=True),
}
done = already_done(RESULTS_CSV, ['stream', 'policy', 'test_year'])
print(f'Already-done folds: {len(done)} (of {len(STREAMS)*len(POLICIES)*len(YEARS)})')

total = len(STREAMS) * len(POLICIES) * len(YEARS)
pbar = tqdm(total=total, desc='LSTM folds')

# Cache phase_ids
_b0 = load_year_bundle(TENSORS_DIR / f'phenology_{YEARS[0]}.npz')
phase_ids = _b0['phase_ids']
del _b0
gc.collect()

for s in STREAMS:
    active = STREAM_CONFIGS[s]
    # rank1 first so the fast folds complete in case of timeout
    for pol_name in ['rank1', 'all10']:
        policy = policy_map[pol_name]
        for test_y in YEARS:
            key = (s, pol_name, int(test_y))
            if key in done:
                pbar.update(1); continue

            t0 = time.time()
            train_b, test_b = load_loyo_for_year(test_y, policy)
            if len(train_b['xid']) == 0 or len(test_b['xid']) == 0:
                del train_b, test_b; gc.collect()
                pbar.update(1); continue

            # Save the model for the All/all10/2020 fold (for F8 logits)
            save_model_for_logits = (s == 'All' and pol_name == 'all10' and test_y == 2020)

            try:
                result = train_one_fold_lstm(
                    train_b, test_b, active,
                    hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    lr=LR, use_amp=USE_AMP, verbose=False,
                    return_model=save_model_for_logits,
                )
            except torch.cuda.OutOfMemoryError as e:
                print(f'  [OOM] {s}/{pol_name}/{test_y}: {e}')
                del train_b, test_b; gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                pbar.update(1); continue

            # If we saved the model, collect full per-day logits for F8
            if save_model_for_logits:
                print(f'  Collecting per-day logits for F8 ({s}/{pol_name}/{test_y})')
                model = result['model']; tb_te = result['tb_te']
                model.eval()
                all_logits = []
                with torch.no_grad():
                    for x, m, y, lm in iter_batches(tb_te, BATCH_SIZE, False, DEVICE):
                        with torch.amp.autocast('cuda', enabled=(USE_AMP and DEVICE.type == 'cuda')):
                            logits = model(x, m, active=active)
                        all_logits.append(logits.cpu().float().numpy())
                logits_arr = np.concatenate(all_logits, axis=0).astype(np.float32)
                pref = f'All__all10__{test_y}'
                np.savez_compressed(
                    LOGITS_NPZ,
                    **{f'{pref}__logits': logits_arr,
                       f'{pref}__yt':     result['truth'].astype(np.int32),
                       f'{pref}__xid':    result['xid'].astype(np.int64),
                       f'{pref}__bag':    result['bag'].astype(np.int64),
                       f'{pref}__rank':   result['rank'].astype(np.int32),
                       f'{pref}__year':   result['year'].astype(np.int32)},
                )
                print(f'  Saved logits {logits_arr.shape} -> {LOGITS_NPZ}')
                del model, tb_te, all_logits, logits_arr
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

            del train_b, test_b
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

            report = evaluate_predictions(
                pred_days=result['pred'].astype(np.float32),
                true_days=result['truth'].astype(np.float32),
                phase_ids=phase_ids,
                years=result['year'], ranks=result['rank'],
            )
            dt = time.time() - t0
            for phase, m in report['per_phase'].iterrows():
                append_row(RESULTS_CSV, {
                    'stream': s, 'policy': pol_name, 'test_year': int(test_y),
                    'phase': int(phase), 'fit_secs': round(dt, 2),
                    **m.to_dict()
                })
            save_pred_npz(PREDS_NPZ, f'{s}__{pol_name}__{test_y}',
                          xid=result['xid'], bag=result['bag'],
                          rank=result['rank'], year=result['year'],
                          yt=result['truth'].astype(np.int32),
                          yp=result['pred'].astype(np.int32))
            macro_mae = report['headline']['macro_mae']
            pbar.set_postfix({'fold': f'{s[:6]}/{pol_name}/{test_y}',
                                'mae': f'{macro_mae:.2f}',
                                'secs': f'{dt:.0f}'})
            pbar.update(1)

pbar.close()
print('\nAll folds processed.')
print(memstat('end:'))


Already-done folds: 0 (of 50)


LSTM folds:   0%|          | 0/50 [00:00<?, ?it/s]

  Saved logits (5181, 426, 7) -> /fs1/home/atikul/Phenology/phase3_outputs/lstm_v2/lstm_logits.npz

All folds processed.
end: RAM=1.03GB  GPU_alloc=0.02GB  GPU_reserved=0.25GB


## 6. Summary tables

In [7]:
results = pd.read_csv(RESULTS_CSV)
print(f'Rows: {len(results)}')

# Headline: macro over phases, mean across LOYO folds
headline = (results.groupby(['stream', 'policy'])
                   .agg(macro_mae=('mae', 'mean'),
                        macro_rmse=('rmse', 'mean'),
                        macro_within_5d=('within_5d', 'mean'))
                   .reset_index())
headline.to_csv(OUTPUT_DIR / 'lstm_v2_headline.csv', index=False)
print('\n=== Headline (macro over phases, mean across LOYO folds) ===')
print(headline.round(3).to_string(index=False))


Rows: 350

=== Headline (macro over phases, mean across LOYO folds) ===
  stream policy  macro_mae  macro_rmse  macro_within_5d
     All  all10      8.309      10.616            0.434
     All  rank1      8.639      12.223            0.420
Met_only  all10     10.089      13.970            0.384
Met_only  rank1      9.805      14.485            0.386
   S1+S2  all10      8.156      10.524            0.433
   S1+S2  rank1     10.460      20.670            0.383
 S1_only  all10      8.351      11.663            0.429
 S1_only  rank1     11.738      24.173            0.370
 S2_only  all10      8.488      10.671            0.408
 S2_only  rank1     40.282      68.174            0.256


In [8]:
# Per-phase × stream table (policy=all10)
sub = results[results['policy'] == 'all10']
table = (sub.groupby(['phase', 'stream'])['mae'].mean()
                .unstack('stream')
                .reindex(['S1_only','S2_only','Met_only','S1+S2','All'], axis=1))
print('\n=== MAE by (phase, stream), all10 policy ===')
print(table.round(2).to_string())



=== MAE by (phase, stream), all10 policy ===
stream  S1_only  S2_only  Met_only  S1+S2    All
phase                                           
10         9.66     9.51     12.83   9.31  10.01
12        10.63    10.47     14.10  10.02  11.97
15         9.59     9.78     10.29   9.80   9.39
18         5.76     6.15      7.11   6.11   5.83
19         8.39     8.16      8.94   7.98   7.57
21         7.70     7.72      8.69   7.25   6.96
24         6.72     7.64      8.66   6.63   6.43


In [10]:
# Policy comparison: shared label vs closest field
pol_compare = (results.groupby(['stream','policy'])['mae'].mean()
                       .unstack('policy'))
pol_compare['Δ (closest − shared)'] = pol_compare['rank1'] - pol_compare['all10']
print('\n=== Policy comparison: macro MAE ===')
print(pol_compare.round(3).to_string())



=== Policy comparison: macro MAE ===
policy     all10   rank1  Δ (closest − shared)
stream                                        
All        8.309   8.639                 0.330
Met_only  10.089   9.805                -0.284
S1+S2      8.156  10.460                 2.305
S1_only    8.351  11.738                 3.387
S2_only    8.488  40.282                31.794


In [9]:
# Per-year × stream table (policy=all10)
sub = results[results['policy'] == 'all10']
per_year = (sub.groupby(['test_year', 'stream'])['mae'].mean()
                .unstack('stream')
                .reindex(['S1_only','S2_only','Met_only','S1+S2','All'], axis=1))
print('\n=== Macro MAE per test year × stream (all10 policy) ===')
print(per_year.round(2).to_string())



=== Macro MAE per test year × stream (all10 policy) ===
stream     S1_only  S2_only  Met_only  S1+S2   All
test_year                                         
2017          8.82     8.19     11.96   7.84  7.95
2018          8.05     8.16     12.29   7.93  8.91
2019          8.05     8.34      8.76   8.10  8.10
2020          8.37     8.10      8.64   8.02  7.91
2021          8.47     9.65      8.80   8.90  8.67


---

## Done. Paste these tables back to me:

1. The **Headline** table from Section 6, cell 1
2. The **MAE by (phase, stream)** table from Section 6, cell 2
3. The **Policy comparison** table from Section 6, cell 3
4. The **Per-year × stream** table from Section 6, cell 4

I will use them to fill in the manuscript v5 numbers and to regenerate the figures.

Optional: also paste the smoke test output from Section 4 so I can confirm the
LSTM is converging properly before the full run.
